# 06 · Statistics and Hypothesis Testing

**Goal:** apply basic statistics to the customer data: descriptive statistics, a confidence interval, correlation and hypothesis tests.

**Source:** `gold.report_customers` (18,482 customers)

**Tests used:** Welch t-test, Mann-Whitney U, one-way ANOVA, chi-square with Cramér's V

**Output:** `statistics_interpretation_notes.csv` in `output/`

## 1. Setup

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import create_engine
from scipy import stats
import config

os.makedirs("output", exist_ok=True)

## 2. Load customers

In [2]:
engine = create_engine(config.get_conn_url())
df = pd.read_sql("SELECT * FROM gold.report_customers", engine)
print(df.shape)

(18482, 15)


## 3. Descriptive statistics

In [3]:
metrics = ["total_sales", "total_order", "avg_order_value", "total_quantity"]
desc = df[metrics].describe().T
desc["iqr"] = desc["75%"] - desc["25%"]
desc["cv"] = desc["std"] / desc["mean"].replace(0, np.nan)
print(desc.round(2))

                   count     mean      std  min   25%    50%     75%      max  \
total_sales      18482.0  1588.10  2123.78  2.0  50.0  272.0  2511.0  13294.0   
total_order      18482.0     1.50     1.10  1.0   1.0    1.0     2.0     28.0   
avg_order_value  18482.0   911.65  1040.61  2.0  40.0  157.0  1770.0   3578.0   
total_quantity   18482.0     3.27     2.63  1.0   2.0    3.0     4.0     68.0   

                    iqr    cv  
total_sales      2461.0  1.34  
total_order         1.0  0.74  
avg_order_value  1730.0  1.14  
total_quantity      2.0  0.81  


Spend is highly skewed: mean total sales is 1,588 against a median of 272 (coefficient of variation 1.34).

## 4. 95% confidence interval for the mean AOV

In [4]:
aov = df["avg_order_value"]
se = stats.sem(aov)
ci_low, ci_high = stats.t.interval(
    confidence=0.95, df=len(aov) - 1, loc=aov.mean(), scale=se
)
mean_aov = aov.mean()
print(f"mean aov: {mean_aov:.2f}")
print(f"95% ci: [{ci_low:.2f},{ci_high:.2f}]")

mean aov: 911.65
95% ci: [896.65,926.66]


This is the mean of customer-level AOV (911.65). It differs from the overall AOV in notebook 08 (total revenue ÷ total orders = 1,061), which weights customers by their number of orders.

## 5. Pearson vs. Spearman: orders vs. total sales

In [5]:
x = df["total_order"]
y = df["total_sales"]
mask = x.notna() & y.notna()
pearson_r, pearson_p = stats.pearsonr(x[mask], y[mask])
spearman_r, spearman_p = stats.spearmanr(x[mask], y[mask])

print(f"pearson r = {pearson_r:.3f},p = {pearson_p:.4g}")
print(f"spearman r = {spearman_r:.3f},p = {spearman_p:.4g}")

pearson r = 0.382,p = 0
spearman r = 0.608,p = 0


Pearson r = 0.382 and Spearman ρ = 0.608 (both p ≈ 0). Orders and total sales are positively related. The gap between the two coefficients reflects the skewed distributions.

## 6. Two-group test: VIP vs. non-VIP average order value

In [33]:
vip = df.loc[df["customer_segments"] == "vip", "avg_order_value"]
non_vip = df.loc[df["customer_segments"] != "vip", "avg_order_value"]
t_stat = stats.ttest_ind(vip, non_vip, equal_var=False)
u_stat = stats.mannwhitneyu(vip, non_vip, alternative="two-sided")
print(f"vip-mean: {vip.mean().round(2)}; non-vip mean: {non_vip.mean():.2f}")
print(f"welch t-test: statistic ={t_stat.statistic:.2f},p_value = {t_stat.pvalue:.4g}")
print(
    f"mann-whitney u: statistic ={u_stat.statistic:.2f},p_value = {u_stat.pvalue:.4g}"
)

vip-mean: 2633.23; non-vip mean: 742.55
welch t-test: statistic =151.70,p_value = 0
mann-whitney u: statistic =26254098.00,p_value = 0


VIP mean AOV is 2,633.23 vs. 742.55 for the rest. Welch t = 151.70 and the Mann-Whitney U test both give p ≈ 0. VIP is defined from total sales, so a higher AOV is expected by construction.

## 7. ANOVA: AOV across customer segments

In [71]:
group = [g["avg_order_value"].values for _, g in df.groupby("customer_segments")]
f_stat, p_value = stats.f_oneway(*group)
print(f"anova f = {f_stat:.3f}, p = {p_value:.4g}")

anova f = 5675.107, p = 0


F = 5,675.1, p ≈ 0: at least one segment has a different mean AOV.

## 8. Chi-square: age group vs. customer segment

In [85]:
table = pd.crosstab(df["age_group"], df["customer_segments"])
chi2, p, dof, expected = stats.chi2_contingency(table)

n = table.to_numpy().sum()
r, k = table.shape
cramers_v = np.sqrt((chi2 / n) / max(1, min(k - 1, r - 1)))
print(table)
print(f"chi-square = {chi2:.3f},do=f{dof},p={p:.4g}")
print(f"cramer's v= {cramers_v:.3f}")

customer_segments    new  regular   vip
age_group                              
40-49               4423      752   461
50 and above       10206     1448  1192
chi-square = 19.875,do=f2,p=4.833e-05
cramer's v= 0.033


χ² = 19.875, p = 4.8e-05 and Cramér's V = 0.033. The association is statistically significant, but the effect size is negligible, so age group says almost nothing about customer segment.

## 9. How to read the results

A short reference table on how to interpret p-values, confidence intervals, effect sizes and causality. It is also saved to `output/`.

In [86]:
stats_notes = pd.DataFrame(
    {
        "topic": ["p-value", "confidence_interval", "effect_size", "causality"],
        "rule": [
            "Evidence against H0 under assumptions",
            "Range describing uncertainty around an estimate",
            "Magnitude of relationship/difference",
            "Correlation or observational tests do not automatically prove causality",
        ],
    }
)
print(stats_notes)
stats_notes.to_csv("output/statistics_interpretation_notes.csv", index=False)

                 topic                                               rule
0              p-value              Evidence against H0 under assumptions
1  confidence_interval    Range describing uncertainty around an estimate
2          effect_size               Magnitude of relationship/difference
3            causality  Correlation or observational tests do not auto...
